<a href="https://colab.research.google.com/github/wmjx691/rental-market-analyzer/blob/main/scraper_basic_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 第一步：環境安裝（Cell 1）

Colab 是 Linux 環境，沒有視窗介面，所以我們必須安裝「無頭模式（Headless）」的瀏覽器驅動。

In [ ]:
# @title 1. 安裝必要套件、瀏覽器驅動與中文字型 (修復方塊字版)
# 安裝 selenium 和 google drive 相關套件
!pip install selenium gspread oauth2client webdriver_manager

# 1. 安裝 Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y

# 2. 關鍵修正：安裝中文字型 (解決截圖方塊字問題)
!apt-get install -y fonts-noto-cjk

print("環境安裝完成！中文字型已部署。")

#### 第二步：Google 權限驗證（Cell 2）

這是 Colab 最強大的地方，不用搞複雜的 API Key，直接用你的 Google 帳號登入驗證，就能讓程式控制你的試算表。
*執行時會跳出視窗要求權限，請點選「允許」。*

In [ ]:
# @title 2. Google 帳號授權與試算表連線
from google.colab import auth
import gspread
from google.auth import default

# 進行身分驗證
auth.authenticate_user()       # 這一行會跳出彈窗要你登入
creds, _ = default()           # 這是暫時性的 Session 憑證
gc = gspread.authorize(creds)

print("Google 帳號授權成功！準備開始爬蟲...")

#### 第三步：爬蟲主程式（Cell 3）

這段程式碼會做兩件事：

1. **爬取目標租屋網**（使用無頭模式，因為 Colab 看不到畫面）。
2. **寫入試算表**：如果檔案不存在，它會自動建立一個名為 `TARGET_SHEET_NAME` 的試算表；如果存在，它會把新資料「附加」在最後面。

In [ ]:
# @title 3. 執行爬蟲並匯入 Google Sheets (時區修正 + 連結 + 自動合併重複廣告版)
import time
import pandas as pd
import re
import pytz # 引入時區套件
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from IPython.display import Image, display

# --- 1. 設定瀏覽器 ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# --- 2. 輔助函式：點擊 ---
def click_element_by_text(driver, text):
    try:
        xpath = f"//label[contains(text(),'{text}')] | //span[contains(text(),'{text}')] | //li[contains(text(),'{text}')]"
        element = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, xpath))
        )
        driver.execute_script("arguments[0].click();", element)
        print(f"-> 成功點擊: {text}")
        time.sleep(1)
        return True
    except:
        return False

# --- 3. 輔助函式：儲存到 Google Sheet ---
def save_to_google_sheet(df, sheet_name='TARGET_SHEET_NAME_v1'):
    if df.empty:
        print("沒有資料可儲存。")
        return

    try:
        sh = gc.open(sheet_name)
        worksheet = sh.sheet1
    except:
        sh = gc.create(sheet_name)
        worksheet = sh.sheet1
        # 如果是新表，寫入標題
        worksheet.append_row(df.columns.tolist())

    # 寫入資料
    values = df.values.tolist()
    worksheet.append_rows(values)
    print(f"✅ 成功寫入 {len(values)} 筆資料到 Google Sheet！")
    print(f"📊 試算表連結: {sh.url}")

# --- 4. 主程式 ---
def run_spider():
    print("啟動爬蟲 (最終進化版)...")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    target_url = "https://rental.example.com.tw/?region=17"
    print(f"前往: {target_url}")
    driver.get(target_url)

    try:
        # 關閉廣告
        try:
            close_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "div.close, i.close, .TIGerm"))
            )
            close_btn.click()
        except: pass

        # 篩選條件
        print("正在篩選條件...")
        time.sleep(2)
        click_element_by_text(driver, "***HIDDEN_District***")
        click_element_by_text(driver, "***HIDDEN_District***")
        click_element_by_text(driver, "整層住家")

        # 點擊搜尋
        print("點擊搜尋...")
        try:
            search_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'搜尋')] | //div[contains(@class,'search')]//button"))
            )
            driver.execute_script("arguments[0].click();", search_btn)
        except: pass

        print("等待載入 & 捲動頁面...")
        time.sleep(5)
        for i in range(4): # 多捲一次確保載入更多
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(1.5)

        # 抓取物件卡片
        items = driver.find_elements(By.CSS_SELECTOR, ".vue-list-rent-item, .listing-recommend-item, div[class*='item']")
        valid_items = [item for item in items if item.size['height'] > 50]

        print(f"🔍 網頁上偵測到 {len(valid_items)} 個物件卡片 (包含重複廣告)")

        raw_data = []

        # 設定台灣時區
        tw_tz = pytz.timezone('Asia/Taipei')
        current_time = datetime.now(tw_tz).strftime("%Y-%m-%d %H:%M:%S")

        for item in valid_items:
            try:
                full_text = item.text
                if len(full_text) < 10: continue

                # --- A. 抓取連結 ---
                link = "N/A"
                try:
                    # 找卡片內的 <a> 標籤
                    link_elm = item.find_element(By.TAG_NAME, "a")
                    link = link_elm.get_attribute("href")
                except: pass

                # --- B. 抓取價格 ---
                price = "N/A"
                price_match = re.search(r'(\d{1,3}(,\d{3})*)\s*元/月', full_text)
                if price_match:
                    price = price_match.group(0).replace("元/月", "").strip() # 只留數字方便分析
                else: continue # 沒價格就跳過

                # --- C. 抓取坪數 ---
                area = "N/A"
                area_match = re.search(r'(\d+\.?\d*)\s*坪', full_text)
                if area_match:
                    area = area_match.group(1)

                # --- D. 抓取標題 ---
                lines = full_text.split('\n')
                title = lines[0] if lines else "N/A"
                if len(title) < 5 and len(lines) > 1: title = lines[1]

                # --- E. 過濾 (保留 20 坪以上) ---
                try:
                    if area != "N/A" and float(area) < 20: continue
                except: pass

                raw_data.append({
                    "抓取時間": current_time,
                    "標題": title,
                    "價格": price,
                    "坪數": area,
                    "連結": link
                })

            except: continue

        if not raw_data:
            print("⚠️ 警告：沒有收集到有效資料。")
            driver.quit()
            return

        # --- F. 資料處理：計算廣告重複投放數 (Pandas Magic) ---
        df = pd.DataFrame(raw_data)

        print(f"📝 原始抓取筆數: {len(df)}")

        # 邏輯：如果「價格」和「坪數」一模一樣，我們就假設它是同一個物件的不同廣告
        # 計算每個物件出現了幾次 (廣告投放數)
        df['廣告投放數'] = df.groupby(['價格', '坪數'])['標題'].transform('count')

        # 去除重複，只保留第一筆，但保留 '廣告投放數' 欄位
        df_unique = df.drop_duplicates(subset=['價格', '坪數'], keep='first')

        print(f"🧹 去除重複廣告後筆數: {len(df_unique)}")

        # 重新排列欄位順序
        df_final = df_unique[['抓取時間', '標題', '價格', '坪數', '廣告投放數', '連結']]

        # 儲存
        save_to_google_sheet(df_final)

    except Exception as e:
        print(f"發生錯誤: {str(e)}")
        driver.save_screenshot('error.png')
        display(Image('error.png'))

    driver.quit()

# 執行
run_spider()

---

### 如何執行你的「一週測試計畫」

既然你要測試一週，且每三天執行一次，操作流程如下：

1. **第一次（今天）：**
* 登入你的 Google Drive，建立一個 Colab 筆記本。
* 將上述三段代碼貼入。
* 依序點擊「播放鍵」執行 Cell 1, 2, 3。
* 執行完後，去你的 Google Drive 根目錄找找看，會有一個 **`TARGET_SHEET_NAME`** 的試算表。打開來確認資料是否正確。


2. **第二次（三天後）：**
* 打開這個 Colab 網頁。
* **重要：** 因為 Colab 會重置環境，所以你必須**再次點擊 Cell 1, 2, 3**。
* 程式會自動把新的資料「新增」到那張試算表的下面，不會覆蓋舊資料。


3. **第三次（六天後）：**
* 重複上述動作。



### 提醒（關於目標租屋網的反爬蟲）

在 Colab 的「無頭模式（Headless）」下，瀏覽器特徵非常明顯，租屋網這種網站有時會直接阻擋（你可能會看到程式跑完但說「抓到 0 筆物件」）。

* **如果發生這種情況**：代表目標租屋網擋掉了 Colab 的 IP 或特徵。這時候最簡單的解法，還是回到我一開始提供的 **PC 本地端執行**（因為你在本地有視窗介面，比較像真人）。
